In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("catalogo", "maintenance_iot")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("golden_schema", "golden")
dbutils.widgets.text("source_table", "sensor_events_clean")
dbutils.widgets.text("target_table", "rig_daily_summary")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
silver_schema = dbutils.widgets.get("silver_schema")
golden_schema = dbutils.widgets.get("golden_schema")
source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")

# ruta = f"abfss://{container}@{datalake}.dfs.core.windows.net/sensor_data.csv"

In [0]:
df_silver = spark.table(f"{catalogo}.{silver_schema}.{source_table}")

print("Registros en Silver:", df_silver.count())

In [0]:
from pyspark.sql.functions import avg, max, min, sum, when, col

df_golden = (
    df_silver
    .groupBy("rig_id", "sensor_type", "ingestion_date")
    .agg(
        avg("value").alias("avg_value"),
        max("value").alias("max_value"),
        min("value").alias("min_value"),
        sum(
            when(col("anomaly_flag") == "ANOMALY", 1).otherwise(0)
        ).alias("anomaly_count")
    )
)

In [0]:
df_golden.write \
    .format("delta") \
    .mode("overwrite") \
    .insertInto(f"{catalogo}.{golden_schema}.{target_table}")

In [0]:
spark.sql(f"""
SELECT *
FROM {catalogo}.{golden_schema}.{target_table}
ORDER BY ingestion_date, rig_id
LIMIT 20
""").show()